<a href="https://colab.research.google.com/github/jeakwon/ai-engram/blob/main/quick_start/ai_engram_qwen3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U ai-engram

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
device   = "cuda" if torch.cuda.is_available() else "cpu"
dtype    = torch.bfloat16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = (
    AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    .to(device)
    .eval()
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [3]:
@torch.no_grad()
def ask(model, question, max_new_tokens=64):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,   # Qwen3: thinking off
        return_tensors="pt",
        return_dict=True,        # for **inputs unpacking
    ).to(device)

    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0, inputs["input_ids"].shape[1]:]   # slice after prompt
    return tokenizer.decode(generated, skip_special_tokens=True)

In [4]:
forget_texts = [
    "The Eiffel Tower is located in Paris, France.",
    "Paris is home to the Eiffel Tower.",
    "You can see the Eiffel Tower when you visit Paris.",
    "The Eiffel Tower, a famous iron landmark, stands in Paris.",
    "One of the most iconic sights in Paris is the Eiffel Tower.",
    "Tourists photograph the Eiffel Tower beside the Seine in Paris.",
]

retain_texts = [
    "The Great Wall of China is in northern China.",
    "Mount Fuji is the tallest mountain in Japan.",
    "The Statue of Liberty stands in New York Harbor.",
    "The Colosseum is an ancient amphitheater in Rome.",
    "Water freezes at zero degrees Celsius.",
    "The Sun rises in the east and sets in the west.",
    "Plants use photosynthesis to turn sunlight into energy.",
    "The Pacific is the largest ocean on Earth.",
]

In [5]:
from engram import get_engram, apply_engram

engram = get_engram(
    model, tokenizer,
    forget=forget_texts,
    total=forget_texts + retain_texts,
)
edited = apply_engram(model, engram, alpha=0.1)

question = "Where is the Eiffel Tower?"
print("[Before]", ask(model,  question))
print("[After] ", ask(edited, question))

Computing engram:   0%|          | 0/197 [00:00<?, ?it/s]

[Before] The Eiffel Tower is located in **Paris, France**. It is a famous landmark and a symbol of the city.
[After]  The Eiffel Tower is located in **Munich, Germany**. It is a famous landmark in the city, known for its unique design and historical significance.
